# FitzHugh–Nagumo model

The FitzHugh–Nagumo model represents a model for the
study of excitable media. The propagation of the transmembrane potential $u$ is modeled by a diffusion equation with a cubic nonlinear reaction term, whereas the recovery of the slow variable $v$ is represented by a single ordinary differential equation. The system of equations reads as follows:

$
\begin{align}
\frac{\partial u}{\partial t} &= D \Delta u + u(1-u)(u-a) -v + S  &&\qquad \text{ for } x \in \Omega \text{ and for } t \in [0, T], \\
\frac{\partial v}{\partial t} &= \epsilon (\beta u - \gamma v) &&\qquad \text{ for } x \in \Omega \text{ and for } t \in [0, T]
\end{align}
$

$\Omega$ denotes the spatial domain and $t$ is time. We consider the following choice of model parameters: $D = 10^{-4}$, $a = 0.1$, $\epsilon = 0.01$,
$\beta = 0.5$, $\gamma = 1.0.$ These parameters generate stable patterns in the system in the form of reentrant spiral waves.

$S$ is a stimulus that we use to perturb the system.

# Semi-discrete problem and weak formulation

At each time step we have the semi-discretized system to solve:

$
\begin{align}
\frac{u^n - u^{n-1}}{\Delta t} &= D \Delta u^n + u^n(1-u^n)(u^n-a) - v^n + S  &&\qquad \text{ for } x \in \Omega \text{ and for } t \in [0, T], \\
\frac{v^n - v^{n-1}}{\Delta t} &= \epsilon (\beta u^n - \gamma v^n) &&\qquad \text{ for } x \in \Omega \text{ and for } t \in [0, T]
\end{align}
$

The weak formulation reads as follows:

Find $u^n$, $v^n$ s.t. 

$
\begin{align}
\langle \frac{u^n - u^{n-1}}{\Delta t}, \psi_u \rangle &= - D \langle \nabla u^n, \nabla \psi_u \rangle + \langle u^n(1-u^n)(u^n-a), \psi_u \rangle - \langle v^n, \psi_u \rangle + \langle S, \psi_u \rangle &&\qquad \text{ for } x \in \Omega \text{ and for } t \in [0, T], \\
\langle \frac{v^n - v^{n-1}}{\Delta t}, \psi_v\rangle &= \epsilon (\beta \langle u^n, \psi_v \rangle - \langle \gamma v^n, \psi_v \rangle ) &&\qquad \text{ for } x \in \Omega \text{ and for } t \in [0, T]  \text{ and for all} \quad \psi_u, \psi_v
\end{align}
$

## FEniCSx implementation

First, we define our domain and finite elements. We have a system of two equations, therefore we will use a Mixed Function Space where the first component is $u$ and the second is $v$.

In [ ]:
from dolfinx import fem, mesh, io, log, plot
from dolfinx.fem.petsc import NonlinearProblem
import basix
from ufl import (
    SpatialCoordinate,
    And,
    lt,
    conditional,
    dx,
    MixedFunctionSpace,
    extract_blocks,
    TestFunctions,
    grad,
    inner,
)
import numpy as np
from mpi4py import MPI
import pyvista as pv
import tqdm.notebook

In [ ]:
# Define the mesh
N = 32
domain = mesh.create_rectangle(
    MPI.COMM_WORLD,
    np.array([[0, 0], [2.5, 2.5]]),
    [N, N],
    cell_type=mesh.CellType.triangle,
)

Ue = basix.ufl.element("Lagrange", domain.basix_cell(), 1)
Vu = fem.functionspace(domain, Ue)
Vv = Vu.clone()
W = MixedFunctionSpace(Vu, Vv)

# Set log level to WARNING to suppress info messages from FEniCSx
log.set_log_level(log.LogLevel.WARNING)

We need to define our stimulus $S$ and boundary conditions: 

In [ ]:
# Find dofs on left boundary
def left_boundary(x):
    return np.isclose(x[0], 0)


dofs_left = fem.locate_dofs_geometrical(Vu, left_boundary)


def BC_U_and_stim(t):
    ## Returns tuple of BCs to apply (to V) and stimulus term at time t
    ## What happens when you play with these parameters?
    start_stim_end_time = 6
    stimstarttime = 150
    stimduration = 6.0
    stimamp = 0.8

    if t < start_stim_end_time:
        # Set a Dirichlet boundary condition of u=1 at left edge of the domain, but no stimulus term S
        bcs = [fem.dirichletbc(1.0, dofs_left, Vu)]
        return bcs, fem.Constant(domain, 0.0)

    elif t > stimstarttime and t < stimstarttime + stimduration:
        # No Dirichlet BCs, stimulus in the bottom left quarter
        x = SpatialCoordinate(domain)
        condition = And(lt(x[0], 1.25), lt(x[1], 1.25))
        stim = conditional(condition, stimamp, 0.0)  # stimamp if True, else 0.0
        return [], stim

    else:
        # No stimulus and no Dirichlet boundary conditions
        return [], fem.Constant(domain, 0.0)

For each time step we need to solve a linear system. We define parameters, trial and test functions, and weak formulation. In FEniCS it is possible to obtain the Jacobian of the nonlinear equations automatically:

In [ ]:
def solve_single_timestep(
    w_p: tuple[fem.Function, ...], W: MixedFunctionSpace, t, dt, theta=1
):
    """
    Solve for a single time step

    Args:
        w_p: Solutions from previous time step
        t: Current simulation time
        dt: Size of time step
        theta: Choose discretization method (1=implicit Euler, 0.5=Crank-Nicholson, 0=Explicit Euler)
    """
    u_p, v_p = w_p
    u, v = fem.Function(u_p.function_space), fem.Function(v_p.function_space)
    psiu, psiv = TestFunctions(W)

    a = fem.Constant(domain, 0.1)
    eps = fem.Constant(domain, 0.01)
    beta = fem.Constant(domain, 0.5)
    gamma = fem.Constant(domain, 1.0)
    D = fem.Constant(domain, 1e-4)
    dtc = fem.Constant(domain, dt)

    bcs, stim = BC_U_and_stim(t)

    # Insert weak formulation for the first equation
    F1 = (u - u_p) / dtc * psiu  # + ...

    # Insert weak formulation for the second equation
    F2 = (v - v_p) / dtc * psiv  # + ...

    F = F1 + F2

    petsc_options = {
        "ksp_type": "preonly",
        "pc_type": "lu",
        "pc_factor_mat_solver_type": "mumps",
        "snes_monitor": None,
    }
    U = [u, v]
    problem = NonlinearProblem(
        extract_blocks(F),
        U,
        bcs=bcs,
        petsc_options=petsc_options,
        petsc_options_prefix="nonlinear_basic",
    )

    # Assign initial guess to Newton solver
    u.interpolate(u_p)
    v.interpolate(v_p)

    # Run Newton solver
    problem.solve()

    return U

Once we have defined the weak formulation and the nonlinear solver we can solve our equations for $t \leq T$:

In [ ]:
# Initialize functions
u = fem.Function(Vu, name="u")
v = fem.Function(Vv, name="v")
U = (u, v)

# Prepare output files
vtx_u = io.VTXWriter(domain.comm, "output/u.bp", u, engine="BP4")
vtx_v = io.VTXWriter(domain.comm, "output/v.bp", v, engine="BP4")

t = 0.0
dt = 4.0
T = 500.0

U_sol = {}
n = 0
total_steps = int(T / dt)

progress_bar_solver = tqdm.notebook.tqdm(
    total=total_steps, desc="Solving time dependent problem"
)


while t <= T:
    t += dt
    n += 1

    print(f"Solving step t = {t}")

    # Solve the timestep
    U = solve_single_timestep(U, W, t, dt)

    # Save solution to dictionary
    u_snapshot = fem.Function(Vu)
    v_snapshot = fem.Function(Vv)
    u_snapshot.interpolate(U[0])
    v_snapshot.interpolate(U[1])
    U_sol[n] = (t, u_snapshot, v_snapshot)

    # Update progress bar
    progress_bar_solver.update(1)

    # Map the mixed solution data into the output functions and save to file
    u.interpolate(U[0])
    v.interpolate(U[1])
    vtx_u.write(t)
    vtx_v.write(t)

vtx_u.close()
vtx_v.close()
progress_bar_solver.close()

In [ ]:
# Generate GIF with Pyvista

# 1. Determine min/max values for clim
u_min = u_max = v_min = v_max = 0
for i in range(1, len(U_sol) + 1):
    u_min = min(u_min, U_sol[i][1].x.array.min())
    u_max = max(u_max, U_sol[i][1].x.array.max())
    v_min = min(v_min, U_sol[i][2].x.array.min())
    v_max = max(v_max, U_sol[i][2].x.array.max())

# 2. Create the PyVista grids
topology_u, cells_u, geometry_u = plot.vtk_mesh(Vu)
grid_u = pv.UnstructuredGrid(topology_u, cells_u, geometry_u)

topology_v, cells_v, geometry_v = plot.vtk_mesh(Vv)
grid_v = pv.UnstructuredGrid(topology_v, cells_v, geometry_v)

# 3. Set up the PyVista Plotter (1 row, 2 columns)
plotter = pv.Plotter(shape=(1, 2), off_screen=True, window_size=[1200, 600])

# Initialize with the first frame to set up the colorbar and mesh
t0, u0, v0 = U_sol[1]
grid_u.point_data["Potential"] = u0.x.array
grid_v.point_data["Recovery"] = v0.x.array

# --- Left subplot (Potential) ---
plotter.subplot(0, 0)
plotter.add_mesh(grid_u, scalars="Potential", cmap="viridis", clim=[u_min, u_max])
plotter.add_text(f"Potential at t={t0:04.1f}", font_size=14, name="title_u")
plotter.view_xy()

# --- Right subplot (Recovery) ---
plotter.subplot(0, 1)
plotter.add_mesh(grid_v, scalars="Recovery", cmap="viridis", clim=[v_min, v_max])
plotter.add_text(f"Recovery at t={t0:04.1f}", font_size=14, name="title_v")
plotter.view_xy()

# 4. Open the GIF
gif_path = "fitzhugh_nagumo.gif"
plotter.open_gif(gif_path, fps=10)

# 5. Render loop
progress_bar = tqdm.notebook.tqdm(total=len(U_sol), desc="Rendering PyVista Video")

for idx in U_sol.keys():
    t, u, v = U_sol[idx]

    # Update the data arrays
    grid_u.point_data["Potential"] = u.x.array
    grid_v.point_data["Recovery"] = v.x.array

    # Update the dynamic text labels
    plotter.subplot(0, 0)
    plotter.add_text(f"Potential at t={t:04.1f}", font_size=14, name="title_u")
    plotter.subplot(0, 1)
    plotter.add_text(f"Recovery at t={t:04.1f}", font_size=14, name="title_v")

    # Write the frame
    plotter.write_frame()
    progress_bar.update(1)

plotter.close()
progress_bar.close()

In [ ]:
# Display the generated GIF
from IPython.display import Image, display

display(Image(gif_path))